# 🎙️ Sleep2K - Thử Nghiệm & Benchmark Voice Cloning Trên Google Colab (GPU T4)

**Mục đích:**
Thử nghiệm độc lập mô hình Voice Cloning (F5-TTS và GPT-SoVITS) trên GPU Tesla T4 (16GB VRAM) hoàn toàn miễn phí của Google Colab.
- Tuyệt đối không cài model AI lên Render (tránh sập RAM 512MB).
- Đo đạc tốc độ xử lý (Real-time factor), độ giống giọng mẫu và độ tự nhiên tiếng Việt / tiếng Trung.
- Sau khi nghe thử kết quả thực tế, chúng ta mới quyết định chọn mô hình tối ưu nhất để tích hợp vào Sleep2K.

## Bước 1: Kiểm tra GPU Tesla T4 (16GB VRAM Miễn Phí)

In [ ]:
!nvidia-smi

## Bước 2: Cài đặt F5-TTS (Zero-shot Voice Cloning)
F5-TTS là công nghệ Non-Autoregressive dựa trên Flow Matching. Ưu điểm là tốc độ sinh rất nhanh và không cần huấn luyện lại.

In [ ]:
!pip install --upgrade pip
!pip install f5-tts gradio torchaudio soundfile librosa
!git clone https://github.com/SWivid/F5-TTS.git

## Bước 3: Upload File Âm Thanh Mẫu (5s - 15s)
Hãy upload 1 file âm thanh mẫu (.wav hoặc .mp3) có giọng người nói rõ ràng, không lẫn nhạc nền.

In [ ]:
from google.colab import files
import os

print("Chọn 1 file audio mẫu (độ dài 5-15 giây) từ máy tính:")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Đã tải lên tệp mẫu: {ref_audio_path}")

## Bước 4: Chạy Thử Nghiệm Nhân Bản Giọng Nói Với F5-TTS & Đo Thời Gian

In [ ]:
import time
import IPython.display as ipd
import os

# Thiết lập nội dung
ref_text = "Nhập nội dung văn bản mà giọng mẫu đã nói trong file audio trên" # Ví dụ: Xin chào mọi người
gen_text = "Xin chào các bạn, đây là giọng nói được nhân bản bằng trí tuệ nhân tạo trên hệ thống Sleep2K. Giọng nói này có độ tự nhiên rất cao và giữ nguyên âm sắc ban đầu."

print("⏳ Đang chạy nhân bản giọng nói...")
start_time = time.time()

# Chạy lệnh CLI infer nhanh của F5-TTS
!f5-tts_infer-cli \
    --model "F5-TTS" \
    --ref_audio "{ref_audio_path}" \
    --ref_text "{ref_text}" \
    --gen_text "{gen_text}" \
    --output_dir "output_test"

elapsed = time.time() - start_time
print(f"\n✅ Hoàn thành trong: {elapsed:.2f} giây!")

# Phát âm thanh kết quả ngay trong Colab
output_file = "output_test/out.wav"
if os.path.exists(output_file):
    ipd.display(ipd.Audio(output_file))
else:
    print("Xem file kết quả trong thư mục output_test")

## Bước 5: Mở Gradio WebUI Để Thử Nghiệm Trực Quan
Lệnh sau sẽ tạo ra một đường link Public URL (Gradio Share) để bạn kéo thả giao diện đồ họa trực tiếp:

In [ ]:
%cd /content/F5-TTS
!python src/f5_tts/infer/infer_gradio.py --share